In [ ]:
%pip install roboflow python-dotenv ultralytics wandb

In [ ]:
import os

from dotenv import load_dotenv

from roboflow import Roboflow

load_dotenv()

ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("caretech-qnw0y").project("v1-caretech-combined-dataset")
dataset = project.version(1).download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to [V1]-CareTech-Combined-Dataset-1 in yolov8:: 100%|██████████| 58572/58572 [00:07<00:00, 7873.52it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# setting up wandb
import wandb

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)


wandb.init(
    entity="caretech",
    # rename this to what u want the run to be named
    project="david",
    name=f"control",
)

invalid escape sequence '\/'
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: msgalvez06 (caretech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# dataset name fix and sanity check
!mv "/content/[V1]-CareTech-Combined-Dataset-1" "/content/V1-CareTech-Combined-Dataset-1"

import os
import glob

for root, dirs, files in os.walk("/content", topdown=True):
    if os.path.basename(root) == "images":
        print("IMAGES FOLDER:", root)
        imgs = glob.glob(os.path.join(root, "*.jpg"))
        print("IMAGES FOLDER:", root, "->", len(imgs))


IMAGES FOLDER: /content/V1-CareTech-Combined-Dataset-1/train/images
IMAGES FOLDER: /content/V1-CareTech-Combined-Dataset-1/train/images -> 20509
IMAGES FOLDER: /content/V1-CareTech-Combined-Dataset-1/test/images
IMAGES FOLDER: /content/V1-CareTech-Combined-Dataset-1/test/images -> 2949
IMAGES FOLDER: /content/V1-CareTech-Combined-Dataset-1/valid/images
IMAGES FOLDER: /content/V1-CareTech-Combined-Dataset-1/valid/images -> 5822


In [ ]:
from ultralytics import YOLO
from wandb.integration.ultralytics import add_wandb_callback


# this is a list of hyperparameters we should be tuning
#
#    mandatory hyperparameters to note:
#    Number of epochs
#    Learning rate
#    Batch size
#    mAP on validation set
#    YOLO filters (if any were used)
#
# i think we should tune
# batch size ("batch" int or float)
# learning rate ("lr0" and "lrf" float)
# momentum ("momentum" float)
# weight decay ("weight_decay" float)
# number of epochs ("epochs" int)
# documenation
# https://docs.ultralytics.com/modes/train/#train-settings


# Load a model
model = YOLO("yolo26m.pt")  # sizes are n, s, m, l, x

add_wandb_callback(model, enable_model_checkpointing=True) # this saves weights to colab

# Train the model
results = model.train(
    # these are the hyperparameters
    model="yolo26m.pt",
    data="/content/V1-CareTech-Combined-Dataset-1/data.yaml",
    epochs=100,
    imgsz=640,
    batch=0.8,
    patience=50,
    cache=True,
    device=0,
    name="" # name of the training run
)

Ultralytics 8.4.10 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=0.8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/V1-CareTech-Combined-Dataset-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience